# NullVector Unified Postgres Cookbook (Specs 01-09)

This notebook replaces the older PostgreSQL cookbooks with one end-to-end walkthrough of the current NullVector surface.

Prerequisites:
- A reachable PostgreSQL database exposed through `NULLVECTOR_POSTGRES_CONNINFO`
- Inline Groq credentials are configured below through `GROQ_API_KEYS`; live requests rotate keys per call when the Groq gateway path is enabled
- Optional model override via `NULLVECTOR_LLM_MODEL` (defaults to `meta-llama/llama-4-scout-17b-16e-instruct`)
- If the live LiteLLM path is unavailable, the notebook falls back to a deterministic noop gateway for the summarized-tree sections

The notebook is safe to rerun because it uses deterministic run ids.


## Section 1 — Setup And Environment

This cookbook is Postgres-backed and keeps as much of the walkthrough deterministic as possible. The only live LLM usage is the summarized Markdown tree path, which now uses Groq with safe per-call key rotation and still preserves the noop fallback when LiteLLM is unavailable.

Runtime observability is now default-on for the NullVector services used below: you should see human-readable progress lines while cells run, and the same structured events are also written to `artifacts/observability/nullvector-events.jsonl` unless you override that destination with `NULLVECTOR_OBSERVABILITY_JSONL_PATH`.

In [17]:
from __future__ import annotations

import json
import os
import sys
from collections import Counter
from pathlib import Path


def banner(title: str) -> None:
    print()
    print("=" * 60)
    print(title)
    print("=" * 60)


def show_json(title: str, payload: object) -> None:
    banner(title)
    print(json.dumps(payload, indent=2, sort_keys=True, default=str))


ROOT = Path.cwd()
print(f"Repository root: {ROOT}")
print(f"Python executable: {sys.executable}")


Repository root: /home/pruthvi/projects/NullVector/cookbook
Python executable: /home/pruthvi/projects/NullVector/.venv/bin/python


## Section 2 — Imports

The imports below intentionally use the current public surface that landed with Specs 01-09, plus the public storage helpers needed to derive PostgreSQL artifact refs. LiteLLM remains optional; the Groq rotation utilities stay inline in this notebook so the runnable path stays self-contained.


In [18]:
from nullvector.domain import (
    AcquisitionRequest,
    DescriptionSelectionRequest,
    DocumentDescriptionRecord,
    DocumentDescriptionRequest,
    DocumentFilterClause,
    DocumentFilterOperator,
    DocumentMetadataRecord,
    DocumentPrefilterRequest,
    DocumentSemanticProxySource,
    MetadataSelectionPlan,
    MetadataSelectionRequest,
    NodeCard,
    PreferenceAwareTreeSearchRequest,
    PreferenceScope,
    PreferenceSnippet,
    SourceDocumentKind,
    TreeBuildRequest,
    TreeCompactionRequest,
    TreeCompactionSettings,
    TreeSearchRequest,
    UnassignedPageSpan,
)
from nullvector.ingest.acquisition_service import AcquisitionService
from nullvector.llm import (
    GatewayAuditConfig,
    GatewayConfig,
    GatewayService,
    NoopProviderAdapter,
    NoopScriptedResponse,
)
from nullvector.observability import (
    DEFAULT_OBSERVABILITY_JSONL_PATH,
    configure_default_runtime_observability,
)
from nullvector.retrieval import (
    DescriptionSelectionService,
    DocumentDescriptionBuilder,
    DocumentSemanticProxyBuilder,
    MetadataSelectionService,
    PreferenceAwareTreeSearchService,
    QueryPlanner,
    RetrievalCorpusBuilder,
    RetrievalQAService,
    RetrievalRanker,
    RetrievalService,
    SemanticPrefilterService,
    TreeSearchService,
    load_document_description,
    load_document_description_manifest,
    load_retrieval_corpus,
)
from nullvector.storage import (
    PostgresStorageConfig,
    build_document_store,
    build_postgres_artifact_ref,
)
from nullvector.tree import (
    TreeCompactionService,
    build_tree,
    expand_serving_node_ids_to_canonical_node_ids,
    load_compacted_node_mappings,
    load_compacted_tree,
)

try:
    import litellm
    from nullvector.llm.adapters import LiteLLMAdapter
except ImportError as exc:  # litellm is an optional dependency
    litellm = None
    LiteLLMAdapter = None
    LITELLM_IMPORT_ERROR = exc
else:
    LITELLM_IMPORT_ERROR = None


## Section 3 — Configuration

We keep two demo documents in play:
- a committed PDF fixture from `fixtures/pdfs/phase01/`
- a Markdown file written into a local notebook temp directory so Spec 05 is part of the same walkthrough

The live summarized-tree path uses the inline `GROQ_API_KEYS` list below and rotates keys once per provider call. Notebook output only shows masked key previews.


In [19]:
COOKBOOK_TMP_ROOT = ROOT / "cookbook" / "_tmp" / "postgres_unified"
COOKBOOK_TMP_ROOT.mkdir(parents=True, exist_ok=True)

PDF_SOURCE_PATH = "/home/pruthvi/projects/NullVector/cookbook/903000608.pdf"
MARKDOWN_SOURCE_PATH = COOKBOOK_TMP_ROOT / "spec05_operating_handbook.md"
MARKDOWN_SOURCE_TEXT = "\n".join(
    (
        "# Operating Handbook",
        "Handbook overview for the quarterly operating playbook.",
        "",
        "## Revenue Policies",
        "Revenue policies describe recognition guardrails and margin reviews.",
        "",
        "## Litigation Policies",
        "Litigation policies describe case deadlines, filing checkpoints, and escalation paths.",
        "",
        "## Customer Support",
        "Customer support runbooks explain triage and escalation routing.",
        "",
        "## Controls Checklist",
        "Controls require approvals, ownership checks, and handoff reviews.",
        "",
        "## Appendix Notes",
        "Appendix notes contain glossary entries and reference links.",
    )
)
MARKDOWN_SOURCE_PATH.write_text(MARKDOWN_SOURCE_TEXT + "\n", encoding="utf-8")

POSTGRES_CONNINFO = os.environ.get(
    "NULLVECTOR_POSTGRES_CONNINFO",
    "postgresql://REDACTED_DB_CRED@localhost:5432/app",
)
POSTGRES_SCHEMA = os.environ.get("NULLVECTOR_POSTGRES_SCHEMA", "public")
GROQ_DEFAULT_MODEL = "meta-llama/llama-4-scout-17b-16e-instruct"
DEFAULT_MODEL = os.environ.get("NULLVECTOR_LLM_MODEL", GROQ_DEFAULT_MODEL)
LITELLM_MODEL = DEFAULT_MODEL if DEFAULT_MODEL.startswith("groq/") else f"groq/{DEFAULT_MODEL}"
USE_GROQ_LIVE_GATEWAY = True


def parse_groq_api_keys(raw_keys: str) -> tuple[str, ...]:
    keys = tuple(part.strip() for part in raw_keys.split(",") if part.strip())
    if not keys:
        raise ValueError("GROQ_API_KEYS must contain at least one non-empty key.")
    return keys


def groq_key_label(slot: int, key: str) -> str:
    suffix = key[-4:] if len(key) >= 4 else key
    return f"slot-{slot:02d} (gsk_...{suffix})"


GROQ_API_KEYS_INLINE = ",".join(
    (
        "REDACTED_GROQ_KEY",
        "REDACTED_GROQ_KEY",
        "REDACTED_GROQ_KEY",
        "REDACTED_GROQ_KEY",
        "REDACTED_GROQ_KEY",
    )
)
GROQ_API_KEYS = parse_groq_api_keys(GROQ_API_KEYS_INLINE)
GROQ_MASKED_KEYS = tuple(
    groq_key_label(slot, key) for slot, key in enumerate(GROQ_API_KEYS, start=1)
)
groq_key_cursor = 0
COLLECTION_ID = "cookbook-postgres-demo"

RUN_IDS = {
    "pdf_acquisition": "cookbook-pdf-acquisition",
    "pdf_tree": "cookbook-pdf-tree",
    "pdf_retrieval": "cookbook-pdf-retrieval",
    "pdf_description": "cookbook-pdf-description",
    "markdown_acquisition": "cookbook-md-acquisition",
    "markdown_tree": "cookbook-md-tree",
    "markdown_retrieval": "cookbook-md-retrieval",
    "markdown_description": "cookbook-md-description",
    "markdown_compaction": "cookbook-md-compaction",
    "metadata_selection": "cookbook-metadata-selection",
    "description_selection": "cookbook-description-selection",
    "semantic_prefilter": "cookbook-semantic-prefilter",
    "tree_search": "cookbook-tree-search",
    "preference_tree_search": "cookbook-preference-tree-search",
}

pg_config = PostgresStorageConfig(conninfo=POSTGRES_CONNINFO, schema=POSTGRES_SCHEMA)
store = build_document_store(pg_config)
OBSERVABILITY_JSONL_PATH = os.environ.get(
    "NULLVECTOR_OBSERVABILITY_JSONL_PATH",
    DEFAULT_OBSERVABILITY_JSONL_PATH,
)
runtime_logger = configure_default_runtime_observability()


def next_groq_key_selection() -> tuple[str, str]:
    global groq_key_cursor
    index = groq_key_cursor % len(GROQ_API_KEYS)
    groq_key_cursor += 1
    return GROQ_API_KEYS[index], GROQ_MASKED_KEYS[index]


def manifest_ref(*, run_type: str, run_id: str, document_id: str) -> str:
    return build_postgres_artifact_ref(
        run_type=run_type,
        run_id=run_id,
        document_id=document_id,
        artifact_path="manifest.json",
    )


def load_node_cards_from_tree_manifest(tree_manifest) -> tuple[NodeCard, ...]:
    if tree_manifest.node_cards_path is None:
        return ()
    payload = store.read_json_artifact(tree_manifest.node_cards_path)
    return tuple(NodeCard.model_validate_json(json.dumps(item)) for item in payload)


def node_title_map(tree_manifest) -> dict[str, str]:
    return {
        card.node_id: card.title
        for card in load_node_cards_from_tree_manifest(tree_manifest)
    }


def load_unassigned_spans_from_tree_manifest(tree_manifest) -> tuple[UnassignedPageSpan, ...]:
    if tree_manifest.unassigned_spans_path is None:
        return ()
    payload = store.read_json_artifact(tree_manifest.unassigned_spans_path)
    return tuple(UnassignedPageSpan.model_validate_json(json.dumps(item)) for item in payload)


def load_artifact_page_count(artifact_path: str | None) -> int:
    if artifact_path is None:
        return 0
    payload = store.read_json_artifact(artifact_path)
    return len(payload["pages"])


def covered_page_indices_from_node_cards(node_cards: tuple[NodeCard, ...]) -> set[int]:
    covered_pages: set[int] = set()
    for card in node_cards:
        covered_pages.update(range(card.page_span.start_page, card.page_span.end_page + 1))
    return covered_pages


def unassigned_page_indices(unassigned_spans: tuple[UnassignedPageSpan, ...]) -> list[int]:
    return sorted(
        {
            page_index
            for span in unassigned_spans
            for page_index in range(span.page_span.start_page, span.page_span.end_page + 1)
        }
    )


def retrieval_unit_type_counts_for_page(corpus, *, page_index: int) -> dict[str, int]:
    counts = Counter(
        unit.unit_type.value
        for unit in corpus.units
        if unit.page_span.start_page <= page_index <= unit.page_span.end_page
    )
    return dict(sorted(counts.items()))


def pdf_coverage_summary(
    acquisition_manifest,
    tree_manifest,
    *,
    retrieval_corpus=None,
) -> dict[str, object]:
    node_cards = load_node_cards_from_tree_manifest(tree_manifest)
    unassigned_spans = load_unassigned_spans_from_tree_manifest(tree_manifest)
    unassigned_pages = unassigned_page_indices(unassigned_spans)
    unassigned_reasons = list(dict.fromkeys(span.reason for span in unassigned_spans))
    return {
        "acquisition_page_count": acquisition_manifest.page_count,
        "ledger_page_count": load_artifact_page_count(acquisition_manifest.ledger_path),
        "substrate_page_count": load_artifact_page_count(
            acquisition_manifest.canonical_text_substrate_path
        ),
        "tree_covered_page_count": len(covered_page_indices_from_node_cards(node_cards)),
        "tree_unassigned_page_count": len(unassigned_pages),
        "tree_unassigned_pages": unassigned_pages,
        "tree_unassigned_reasons": unassigned_reasons,
        "page0_retrieval_unit_types": (
            retrieval_unit_type_counts_for_page(retrieval_corpus, page_index=0)
            if retrieval_corpus is not None
            else {}
        ),
    }


def build_demo_gateway() -> tuple[GatewayService, str]:
    audit_root = str(COOKBOOK_TMP_ROOT / "gateway_audit")
    if USE_GROQ_LIVE_GATEWAY and LiteLLMAdapter is not None and litellm is not None:
        def rotating_completion(**kwargs):
            api_key, label = next_groq_key_selection()
            print(f"Groq rotation -> {label}")
            kwargs["api_key"] = api_key
            return litellm.completion(**kwargs)

        try:
            return (
                GatewayService(
                    GatewayConfig(
                        default_model=LITELLM_MODEL,
                        audit=GatewayAuditConfig(persist_root=audit_root),
                    ),
                    provider_adapter=LiteLLMAdapter(completion_fn=rotating_completion),
                    logger=runtime_logger,
                ),
                "litellm-groq-round-robin",
            )
        except Exception as exc:
            print(f"Falling back to NoopProviderAdapter: {exc}")
    elif USE_GROQ_LIVE_GATEWAY and LITELLM_IMPORT_ERROR is not None:
        print(f"LiteLLMAdapter unavailable; using noop gateway: {LITELLM_IMPORT_ERROR}")

    scripts = {
        "summarize_leaf_node": NoopScriptedResponse(
            output_json={
                "summary": "Deterministic notebook leaf summary.",
                "keywords": ["policy", "litigation", "revenue"],
            }
        ),
        "summarize_parent_node": NoopScriptedResponse(
            output_json={
                "summary": "Deterministic notebook parent summary.",
                "keywords": ["handbook", "policy", "summary"],
            }
        ),
    }
    return (
        GatewayService(
            GatewayConfig(
                default_model="noop-model",
                audit=GatewayAuditConfig(persist_root=audit_root),
            ),
            provider_adapter=NoopProviderAdapter(scripts),
            logger=runtime_logger,
        ),
        "noop-scripted",
    )


gateway, gateway_mode = build_demo_gateway()

show_json(
    "Notebook configuration",
    {
        "pdf_source_path": str(PDF_SOURCE_PATH),
        "markdown_source_path": str(MARKDOWN_SOURCE_PATH),
        "postgres_schema": POSTGRES_SCHEMA,
        "default_model": DEFAULT_MODEL,
        "litellm_model": LITELLM_MODEL,
        "groq_key_count": len(GROQ_API_KEYS),
        "groq_keys": list(GROQ_MASKED_KEYS),
        "gateway_mode": gateway_mode,
        "collection_id": COLLECTION_ID,
        "observability_jsonl_path": OBSERVABILITY_JSONL_PATH,
        "observability_enabled": runtime_logger is not None,
    },
)



Notebook configuration
{
  "collection_id": "cookbook-postgres-demo",
  "default_model": "meta-llama/llama-4-scout-17b-16e-instruct",
  "gateway_mode": "litellm-groq-round-robin",
  "groq_key_count": 5,
  "groq_keys": [
    "slot-01 (gsk_...AnOM)",
    "slot-02 (gsk_...xhRt)",
    "slot-03 (gsk_...74oG)",
    "slot-04 (gsk_...x09a)",
    "slot-05 (gsk_...sDUk)"
  ],
  "litellm_model": "groq/meta-llama/llama-4-scout-17b-16e-instruct",
  "markdown_source_path": "/home/pruthvi/projects/NullVector/cookbook/cookbook/_tmp/postgres_unified/spec05_operating_handbook.md",
  "observability_enabled": true,
  "observability_jsonl_path": "artifacts/observability/nullvector-events.jsonl",
  "pdf_source_path": "/home/pruthvi/projects/NullVector/cookbook/903000608.pdf",
  "postgres_schema": "public"
}


## Section 4 — Acquisition (Spec 05: PDF and Markdown)


In [20]:
acquisition_service = AcquisitionService(logger=runtime_logger, storage=pg_config)

pdf_acquisition = acquisition_service.acquire(
    AcquisitionRequest(
        source_path=str(PDF_SOURCE_PATH),
        acquisition_run_id=RUN_IDS["pdf_acquisition"],
        source_kind=SourceDocumentKind.PDF,
        provider_identity="native_pymupdf",
    )
)
markdown_acquisition = acquisition_service.acquire(
    AcquisitionRequest(
        source_path=str(MARKDOWN_SOURCE_PATH),
        acquisition_run_id=RUN_IDS["markdown_acquisition"],
        source_kind=SourceDocumentKind.MARKDOWN,
        provider_identity="markdown_native",
    )
)

pdf_acquisition_manifest_path = manifest_ref(
    run_type="acquisition",
    run_id=RUN_IDS["pdf_acquisition"],
    document_id=pdf_acquisition.document_id,
)
markdown_acquisition_manifest_path = manifest_ref(
    run_type="acquisition",
    run_id=RUN_IDS["markdown_acquisition"],
    document_id=markdown_acquisition.document_id,
)

show_json(
    "Acquisition manifests",
    {
        "pdf": {
            "document_id": pdf_acquisition.document_id,
            "page_count": pdf_acquisition.page_count,
            "selected_outline_source": pdf_acquisition.selected_outline_source,
            "manifest_ref": pdf_acquisition_manifest_path,
        },
        "markdown": {
            "document_id": markdown_acquisition.document_id,
            "page_count": markdown_acquisition.page_count,
            "selected_outline_source": markdown_acquisition.selected_outline_source,
            "manifest_ref": markdown_acquisition_manifest_path,
        },
    },
)


[SourceFingerprintComputed] document=798d2f27d45d2ccda3694005c2ed60bc0b413b8b299f3a5d4ade7c5867094896
[AcquisitionStarted] document=798d2f27d45d2ccda3694005c2ed60bc0b413b8b299f3a5d4ade7c5867094896 run=cookbook-pdf-acquisition provider=native_pymupdf


[SourceFingerprintComputed] document=e386599943c9455ea1113d460e077bef156f7e0b738f6052be99f90592feab23
[AcquisitionStarted] document=e386599943c9455ea1113d460e077bef156f7e0b738f6052be99f90592feab23 run=cookbook-md-acquisition provider=markdown_native



Acquisition manifests
{
  "markdown": {
    "document_id": "e386599943c9455ea1113d460e077bef156f7e0b738f6052be99f90592feab23",
    "manifest_ref": "pg://acquisition/cookbook-md-acquisition/e386599943c9455ea1113d460e077bef156f7e0b738f6052be99f90592feab23/manifest.json",
    "page_count": 1,
    "selected_outline_source": "markdown"
  },
  "pdf": {
    "document_id": "798d2f27d45d2ccda3694005c2ed60bc0b413b8b299f3a5d4ade7c5867094896",
    "manifest_ref": "pg://acquisition/cookbook-pdf-acquisition/798d2f27d45d2ccda3694005c2ed60bc0b413b8b299f3a5d4ade7c5867094896/manifest.json",
    "page_count": 148,
    "selected_outline_source": "pymupdf"
  }
}


## Section 5 — Tree Build

The PDF summary below separates acquisition coverage from verified node coverage. For `903000608.pdf`, page 0 is a visual-only front page, so it remains an explicit `before_first_heading` gap instead of being attached to a synthetic front-matter node.


In [21]:
pdf_tree = build_tree(
    TreeBuildRequest(
        acquisition_manifest_path=pdf_acquisition_manifest_path,
        tree_run_id=RUN_IDS["pdf_tree"],
        summarize=False,
    ),
    logger=runtime_logger,
    storage=pg_config,
)
markdown_tree = build_tree(
    TreeBuildRequest(
        acquisition_manifest_path=markdown_acquisition_manifest_path,
        tree_run_id=RUN_IDS["markdown_tree"],
        summarize=True,
    ),
    gateway=gateway,
    logger=runtime_logger,
    storage=pg_config,
)

pdf_tree_manifest_path = manifest_ref(
    run_type="tree",
    run_id=RUN_IDS["pdf_tree"],
    document_id=pdf_tree.document_id,
)
markdown_tree_manifest_path = manifest_ref(
    run_type="tree",
    run_id=RUN_IDS["markdown_tree"],
    document_id=markdown_tree.document_id,
)

pdf_tree_coverage = pdf_coverage_summary(pdf_acquisition, pdf_tree)

show_json(
    "Tree build summaries",
    {
        "pdf": {
            "tree_run_id": pdf_tree.tree_run_id,
            "committed_node_count": pdf_tree.committed_node_count,
            "coverage": pdf_tree_coverage,
            "node_card_titles": [card.title for card in load_node_cards_from_tree_manifest(pdf_tree)[:6]],
            "manifest_ref": pdf_tree_manifest_path,
        },
        "markdown": {
            "tree_run_id": markdown_tree.tree_run_id,
            "committed_node_count": markdown_tree.committed_node_count,
            "node_card_titles": [card.title for card in load_node_cards_from_tree_manifest(markdown_tree)[:6]],
            "manifest_ref": markdown_tree_manifest_path,
            "gateway_mode": gateway_mode,
        },
    },
)



Tree build summaries
{
  "markdown": {
    "committed_node_count": 6,
    "gateway_mode": "litellm-groq-round-robin",
    "manifest_ref": "pg://tree/cookbook-md-tree/e386599943c9455ea1113d460e077bef156f7e0b738f6052be99f90592feab23/manifest.json",
    "node_card_titles": [
      "Operating Handbook",
      "Revenue Policies",
      "Litigation Policies",
      "Customer Support",
      "Controls Checklist",
      "Appendix Notes"
    ],
    "tree_run_id": "cookbook-md-tree"
  },
  "pdf": {
    "committed_node_count": 3000,
    "coverage": {
      "acquisition_page_count": 148,
      "ledger_page_count": 148,
      "page0_retrieval_unit_types": {},
      "substrate_page_count": 148,
      "tree_covered_page_count": 147,
      "tree_unassigned_page_count": 1,
      "tree_unassigned_pages": [
        0
      ],
      "tree_unassigned_reasons": [
        "before_first_heading"
      ]
    },
    "manifest_ref": "pg://tree/cookbook-pdf-tree/798d2f27d45d2ccda3694005c2ed60bc0b413b8b299f3a5d4a

## Section 6 — Tree Compaction (Spec 08)


In [22]:
# max_children_per_node=2 is intentionally aggressive for demonstration purposes.
# The Markdown fixture has 5 sections under root, so this forces compaction to fire
# and produce synthetic serving nodes. Production values are typically 8-15.
markdown_compaction = TreeCompactionService(storage=pg_config).compact(
    TreeCompactionRequest(
        tree_manifest_path=markdown_tree_manifest_path,
        compaction_run_id=RUN_IDS["markdown_compaction"],
        settings=TreeCompactionSettings(max_children_per_node=2),
    )
)

compacted_tree = load_compacted_tree(
    markdown_compaction.compacted_tree_path,
    storage=pg_config,
)
compacted_mappings = load_compacted_node_mappings(
    markdown_compaction.node_mapping_path,
    storage=pg_config,
)
example_serving_node = next(
    (node for node in compacted_tree if "::compact::" in node.serving_node_id),
    compacted_tree[0],
)
expanded_canonical_ids = expand_serving_node_ids_to_canonical_node_ids(
    (example_serving_node.serving_node_id,),
    compacted_mappings,
)

show_json(
    "Compaction summary",
    {
        "tree_run_id": markdown_compaction.tree_run_id,
        "compaction_run_id": markdown_compaction.compaction_run_id,
        "canonical_node_count": markdown_tree.committed_node_count,
        "compacted_node_count": len(compacted_tree),
        "example_serving_node_id": example_serving_node.serving_node_id,
        "example_serving_title": example_serving_node.title,
        "example_canonical_node_ids": list(expanded_canonical_ids),
        "manifest_ref": manifest_ref(
            run_type="tree_compaction",
            run_id=RUN_IDS["markdown_compaction"],
            document_id=markdown_compaction.document_id,
        ),
    },
)


Compaction summary
{
  "canonical_node_count": 6,
  "compacted_node_count": 3,
  "compaction_run_id": "cookbook-md-compaction",
  "example_canonical_node_ids": [
    "009a6902c2815ed71814dd6edb118f9cd0abe04b8887f5baf778bd8b53f442f1",
    "0e3949c76a91e8bac56340671f1960380909f658e5c0b783955c92d2d2ac8189",
    "1337ac7faf63539a6c3e073b467b2c268aacdf5069bd0065dbe1ec9dcd50aec4",
    "f0773b053dbd186567bd8b0ec53600028fb583fdbb397a53a84b8ef3382cfe87"
  ],
  "example_serving_node_id": "0fdecb35a49c0bf1af26897e9b8ef6c23ab0e501345338ae3ca2cc3e8409226a::compact::01",
  "example_serving_title": "Appendix Notes (+3)",
  "manifest_ref": "pg://tree_compaction/cookbook-md-compaction/e386599943c9455ea1113d460e077bef156f7e0b738f6052be99f90592feab23/manifest.json",
  "tree_run_id": "cookbook-md-tree"
}


## Section 7 — Retrieval Corpus

Explicit tree gaps are still preserved in retrieval. For the PDF demo, the visual-only front page remains queryable through `unassigned_span` and `visual` units even though it sits outside committed heading nodes.


In [23]:
retrieval_builder = RetrievalCorpusBuilder(logger=runtime_logger, storage=pg_config)

pdf_retrieval = retrieval_builder.build(
    acquisition_manifest_path=pdf_acquisition_manifest_path,
    tree_manifest_path=pdf_tree_manifest_path,
    retrieval_run_id=RUN_IDS["pdf_retrieval"],
)
markdown_retrieval = retrieval_builder.build(
    acquisition_manifest_path=markdown_acquisition_manifest_path,
    tree_manifest_path=markdown_tree_manifest_path,
    retrieval_run_id=RUN_IDS["markdown_retrieval"],
)

pdf_corpus = load_retrieval_corpus(
    pdf_retrieval.corpus_path,
    storage=pg_config,
)
markdown_corpus = load_retrieval_corpus(
    markdown_retrieval.corpus_path,
    storage=pg_config,
)
pdf_counts = Counter(unit.unit_type.value for unit in pdf_corpus.units)
markdown_counts = Counter(unit.unit_type.value for unit in markdown_corpus.units)
pdf_tree_coverage = pdf_coverage_summary(
    pdf_acquisition,
    pdf_tree,
    retrieval_corpus=pdf_corpus,
)

show_json(
    "Retrieval corpus summary",
    {
        "pdf": {
            "document_id": pdf_retrieval.document_id,
            "unit_count": pdf_retrieval.unit_count,
            "counts_by_type": dict(sorted(pdf_counts.items())),
            "coverage": pdf_tree_coverage,
            "corpus_ref": pdf_retrieval.corpus_path,
            "manifest_ref": manifest_ref(
                run_type="retrieval",
                run_id=RUN_IDS["pdf_retrieval"],
                document_id=pdf_retrieval.document_id,
            ),
        },
        "markdown": {
            "document_id": markdown_retrieval.document_id,
            "unit_count": markdown_retrieval.unit_count,
            "counts_by_type": dict(sorted(markdown_counts.items())),
            "corpus_ref": markdown_retrieval.corpus_path,
            "manifest_ref": manifest_ref(
                run_type="retrieval",
                run_id=RUN_IDS["markdown_retrieval"],
                document_id=markdown_retrieval.document_id,
            ),
        },
    },
)


[RetrievalCorpusBuildStarted] document=798d2f27d45d2ccda3694005c2ed60bc0b413b8b299f3a5d4ade7c5867094896 tree=cookbook-pdf-tree retrieval=cookbook-pdf-retrieval
[RetrievalCorpusBuildStarted] document=e386599943c9455ea1113d460e077bef156f7e0b738f6052be99f90592feab23 tree=cookbook-md-tree retrieval=cookbook-md-retrieval



Retrieval corpus summary
{
  "markdown": {
    "corpus_ref": "pg://retrieval/cookbook-md-retrieval/e386599943c9455ea1113d460e077bef156f7e0b738f6052be99f90592feab23/corpus.json",
    "counts_by_type": {
      "node_summary": 6,
      "node_text": 6,
      "page_text": 1
    },
    "document_id": "e386599943c9455ea1113d460e077bef156f7e0b738f6052be99f90592feab23",
    "manifest_ref": "pg://retrieval/cookbook-md-retrieval/e386599943c9455ea1113d460e077bef156f7e0b738f6052be99f90592feab23/manifest.json",
    "unit_count": 13
  },
  "pdf": {
    "corpus_ref": "pg://retrieval/cookbook-pdf-retrieval/798d2f27d45d2ccda3694005c2ed60bc0b413b8b299f3a5d4ade7c5867094896/corpus.json",
    "counts_by_type": {
      "node_text": 3000,
      "page_text": 140,
      "table": 46,
      "unassigned_span": 1,
      "unresolved_visual": 4,
      "visual": 197
    },
    "coverage": {
      "acquisition_page_count": 148,
      "ledger_page_count": 148,
      "page0_retrieval_unit_types": {
        "unassigned_s

## Section 8 — Document Descriptions (Spec 01)


In [24]:
description_builder = DocumentDescriptionBuilder(logger=runtime_logger, storage=pg_config)

pdf_description_manifest = description_builder.build(
    DocumentDescriptionRequest(
        acquisition_manifest_path=pdf_acquisition_manifest_path,
        tree_manifest_path=pdf_tree_manifest_path,
        description_run_id=RUN_IDS["pdf_description"],
    )
)
markdown_description_manifest = description_builder.build(
    DocumentDescriptionRequest(
        acquisition_manifest_path=markdown_acquisition_manifest_path,
        tree_manifest_path=markdown_tree_manifest_path,
        description_run_id=RUN_IDS["markdown_description"],
    )
)

pdf_description_manifest_ref = manifest_ref(
    run_type="document_description",
    run_id=RUN_IDS["pdf_description"],
    document_id=pdf_description_manifest.document_id,
)
markdown_description_manifest_ref = manifest_ref(
    run_type="document_description",
    run_id=RUN_IDS["markdown_description"],
    document_id=markdown_description_manifest.document_id,
)

pdf_description_manifest_loaded = load_document_description_manifest(
    pdf_description_manifest_ref,
    storage=pg_config,
)
markdown_description_manifest_loaded = load_document_description_manifest(
    markdown_description_manifest_ref,
    storage=pg_config,
)
pdf_description = load_document_description(
    pdf_description_manifest_loaded.description_path,
    storage=pg_config,
)
markdown_description = load_document_description(
    markdown_description_manifest_loaded.description_path,
    storage=pg_config,
)

show_json(
    "Document descriptions",
    {
        "pdf": {
            "description_method": pdf_description.description_method,
            "description_text": pdf_description.description_text,
            "source_node_ids": list(pdf_description.source_node_ids),
            "manifest_ref": pdf_description_manifest_ref,
        },
        "markdown": {
            "description_method": markdown_description.description_method,
            "description_text": markdown_description.description_text,
            "source_node_ids": list(markdown_description.source_node_ids),
            "manifest_ref": markdown_description_manifest_ref,
        },
    },
)


[DocumentDescriptionBuildStarted] document=798d2f27d45d2ccda3694005c2ed60bc0b413b8b299f3a5d4ade7c5867094896 tree=cookbook-pdf-tree description=cookbook-pdf-description
[DocumentDescriptionBuildStarted] document=e386599943c9455ea1113d460e077bef156f7e0b738f6052be99f90592feab23 tree=cookbook-md-tree description=cookbook-md-description



Document descriptions
{
  "markdown": {
    "description_method": "deterministic_fallback",
    "description_text": "Sections include Operating Handbook, Appendix Notes, Controls Checklist, Customer Support, Litigation Policies, Revenue Policies. The Operating Handbook provides an overview for the quarterly operating playbook, covering key areas such as revenue policies, litigation policies, customer support, controls checklist, and appendix notes.",
    "manifest_ref": "pg://document_description/cookbook-md-description/e386599943c9455ea1113d460e077bef156f7e0b738f6052be99f90592feab23/manifest.json",
    "source_node_ids": [
      "0fdecb35a49c0bf1af26897e9b8ef6c23ab0e501345338ae3ca2cc3e8409226a",
      "009a6902c2815ed71814dd6edb118f9cd0abe04b8887f5baf778bd8b53f442f1",
      "0e3949c76a91e8bac56340671f1960380909f658e5c0b783955c92d2d2ac8189",
      "1337ac7faf63539a6c3e073b467b2c268aacdf5069bd0065dbe1ec9dcd50aec4",
      "f0773b053dbd186567bd8b0ec53600028fb583fdbb397a53a84b8ef3382cfe87

## Section 9 — Collection Selection Before Retrieval

Three selection strategies narrow a multi-document collection before retrieval:
- **Spec 02 — Metadata Selection**: deterministic attribute filters (EQ, CONTAINS)
- **Spec 03 — Description Selection**: ranks documents by description similarity to a query
- **Spec 04 — Semantic Prefilter**: matches against multi-field semantic proxies built from descriptions and node titles

### 9a — Metadata Selection (Spec 02)

In [25]:
metadata_records = (
    DocumentMetadataRecord(
        document_id=pdf_retrieval.document_id,
        display_name="Born Digital Outline PDF",
        attributes={
            "source_kind": "pdf",
            "company": "NullVector",
            "year": 2024,
            "topic": "outline demo and born-digital reference",
        },
    ),
    DocumentMetadataRecord(
        document_id=markdown_retrieval.document_id,
        display_name="Operating Handbook Markdown",
        attributes={
            "source_kind": "markdown",
            "company": "Policy Labs",
            "year": 2025,
            "topic": "operating handbook policies and litigation deadlines",
        },
    ),
)

metadata_response = MetadataSelectionService(logger=runtime_logger, storage=pg_config).select(
    MetadataSelectionRequest(
        collection_id=COLLECTION_ID,
        selection_run_id=RUN_IDS["metadata_selection"],
        allowed_fields=("source_kind", "company", "year", "topic"),
        metadata_records=metadata_records,
        plan=MetadataSelectionPlan(
            raw_query="markdown policy handbook",
            normalized_query="markdown policy handbook",
            clauses=(
                DocumentFilterClause(
                    field="source_kind",
                    operator=DocumentFilterOperator.EQ,
                    value="markdown",
                ),
                DocumentFilterClause(
                    field="topic",
                    operator=DocumentFilterOperator.CONTAINS,
                    value="policy",
                ),
            ),
        ),
        limit=2,
    )
)

show_json(
    "Metadata selection (Spec 02)",
    {
        "candidate_ids": [candidate.document_id for candidate in metadata_response.candidates],
        "matched_metadata": [candidate.matched_metadata for candidate in metadata_response.candidates],
        "results_ref": metadata_response.selection_results_path,
    },
)

[MetadataSelectionStarted] collection=cookbook-postgres-demo selection=cookbook-metadata-selection clauses=2 records=2
[MetadataSelectionCompleted] collection=cookbook-postgres-demo selection=cookbook-metadata-selection candidates=0 clauses=2 records=2



Metadata selection (Spec 02)
{
  "candidate_ids": [],
  "matched_metadata": [],
  "results_ref": "pg://document_selection/cookbook-metadata-selection/cookbook-postgres-demo/document-selection/cookbook-postgres-demo/cookbook-metadata-selection/metadata-selection-results.json"
}


### 9b — Description Selection (Spec 03)

In [26]:
description_records = (
    DocumentDescriptionRecord(
        document_id=pdf_retrieval.document_id,
        display_name="Born Digital Outline PDF",
        description_text=pdf_description.description_text,
        description_manifest_path=pdf_description_manifest_ref,
    ),
    DocumentDescriptionRecord(
        document_id=markdown_retrieval.document_id,
        display_name="Operating Handbook Markdown",
        description_text=markdown_description.description_text,
        description_manifest_path=markdown_description_manifest_ref,
    ),
)
description_selection_response = DescriptionSelectionService(logger=runtime_logger, storage=pg_config).select(
    DescriptionSelectionRequest(
        collection_id=COLLECTION_ID,
        selection_run_id=RUN_IDS["description_selection"],
        query="Which document focuses on operating handbook policies and deadlines?",
        descriptions=description_records,
        limit=2,
    )
)

show_json(
    "Description selection (Spec 03)",
    {
        "candidate_ids": [candidate.document_id for candidate in description_selection_response.candidates],
        "scores": [candidate.score for candidate in description_selection_response.candidates],
        "results_ref": description_selection_response.selection_results_path,
    },
)

[DescriptionSelectionStarted] query="Which document focuses on operating handbook policies and deadlines?" collection=cookbook-postgres-demo selection=cookbook-description-selection descriptions=2
[DescriptionSelectionCompleted] query="Which document focuses on operating handbook policies and deadlines?" collection=cookbook-postgres-demo selection=cookbook-description-selection mode=deterministic_fallback candidates=2 descriptions=2



Description selection (Spec 03)
{
  "candidate_ids": [
    "e386599943c9455ea1113d460e077bef156f7e0b738f6052be99f90592feab23",
    "798d2f27d45d2ccda3694005c2ed60bc0b413b8b299f3a5d4ade7c5867094896"
  ],
  "results_ref": "pg://document_selection/cookbook-description-selection/cookbook-postgres-demo/document-selection/cookbook-postgres-demo/cookbook-description-selection/description-selection-results.json",
  "scores": [
    0.4444444444444444,
    0.0
  ]
}


### 9c — Semantic Prefilter (Spec 04)

In [27]:
semantic_proxy_sources = (
    DocumentSemanticProxySource(
        document_id=pdf_retrieval.document_id,
        display_name="Born Digital Outline PDF",
        description_manifest_path=pdf_description_manifest_ref,
        tree_manifest_path=pdf_tree_manifest_path,
    ),
    DocumentSemanticProxySource(
        document_id=markdown_retrieval.document_id,
        display_name="Operating Handbook Markdown",
        description_manifest_path=markdown_description_manifest_ref,
        tree_manifest_path=markdown_tree_manifest_path,
    ),
)
semantic_proxies = DocumentSemanticProxyBuilder(logger=runtime_logger, storage=pg_config).build(
    semantic_proxy_sources,
)
semantic_prefilter_response = SemanticPrefilterService(logger=runtime_logger, storage=pg_config).select(
    DocumentPrefilterRequest(
        collection_id=COLLECTION_ID,
        selection_run_id=RUN_IDS["semantic_prefilter"],
        query="litigation policies and case deadlines",
        proxies=semantic_proxies,
        limit=2,
    )
)

show_json(
    "Semantic prefilter (Spec 04)",
    {
        "document_ids": [hit.document_id for hit in semantic_prefilter_response.hits],
        "matched_proxy_fields": [list(hit.matched_proxy_fields) for hit in semantic_prefilter_response.hits],
        "results_ref": semantic_prefilter_response.semantic_prefilter_results_path,
    },
)

[SemanticProxyBuildStarted] candidates=2
[SemanticProxyBuildCompleted] candidates=2 proxies=2
[SemanticPrefilterStarted] query="litigation policies and case deadlines" collection=cookbook-postgres-demo selection=cookbook-semantic-prefilter proxies=2
[SemanticPrefilterCompleted] query="litigation policies and case deadlines" collection=cookbook-postgres-demo selection=cookbook-semantic-prefilter hits=2 proxies=2



Semantic prefilter (Spec 04)
{
  "document_ids": [
    "e386599943c9455ea1113d460e077bef156f7e0b738f6052be99f90592feab23",
    "798d2f27d45d2ccda3694005c2ed60bc0b413b8b299f3a5d4ade7c5867094896"
  ],
  "matched_proxy_fields": [
    [
      "description_text",
      "summary_text"
    ],
    []
  ],
  "results_ref": "pg://document_selection/cookbook-semantic-prefilter/cookbook-postgres-demo/document-selection/cookbook-postgres-demo/cookbook-semantic-prefilter/semantic-prefilter-results.json"
}


## Section 10 — Tree Search (Spec 06)


In [28]:
planner = QueryPlanner()
ranker = RetrievalRanker()
retrieval_service = RetrievalService(planner, ranker, logger=runtime_logger, storage=pg_config)
tree_search_service = TreeSearchService(
    planner,
    retrieval_service,
    logger=runtime_logger,
    storage=pg_config,
)
markdown_titles_by_id = node_title_map(markdown_tree)

tree_search_response = tree_search_service.search(
    TreeSearchRequest(
        query="litigation policies",
        tree_manifest_path=markdown_tree_manifest_path,
        retrieval_manifest_path=manifest_ref(
            run_type="retrieval",
            run_id=RUN_IDS["markdown_retrieval"],
            document_id=markdown_retrieval.document_id,
        ),
        search_run_id=RUN_IDS["tree_search"],
        max_selected_nodes=1,
        retrieval_limit=5,
    )
)

show_json(
    "Tree search response",
    {
        "search_mode": tree_search_response.search_mode,
        "selected_node_titles": [
            markdown_titles_by_id.get(candidate.node_id, candidate.node_id)
            for candidate in tree_search_response.selected_nodes
        ],
        "trace": [step.model_dump(mode="json") for step in tree_search_response.trace],
        "retrieval_hit_ids": [hit.unit.unit_id for hit in tree_search_response.retrieval_hits],
        "results_ref": tree_search_response.results_path,
    },
)


[TreeSearchStarted] query="litigation policies" document=e386599943c9455ea1113d460e077bef156f7e0b738f6052be99f90592feab23 tree=cookbook-md-tree search=cookbook-tree-search mode=deterministic max_depth=4
[RetrievalSearchStarted] query="litigation policies" document=e386599943c9455ea1113d460e077bef156f7e0b738f6052be99f90592feab23 limit=5
[RetrievalSearchCompleted] query="litigation policies" document=e386599943c9455ea1113d460e077bef156f7e0b738f6052be99f90592feab23 limit=5 candidates=3 hits=3 widened=false
[TreeSearchStepSelected] document=e386599943c9455ea1113d460e077bef156f7e0b738f6052be99f90592feab23 tree=cookbook-md-tree search=cookbook-tree-search step=0 frontier=1 selected_nodes=1
[TreeSearchCompleted] document=e386599943c9455ea1113d460e077bef156f7e0b738f6052be99f90592feab23 tree=cookbook-md-tree search=cookbook-tree-search mode=deterministic hits=3 selected=1



Tree search response
{
  "results_ref": "pg://tree_search/cookbook-tree-search/e386599943c9455ea1113d460e077bef156f7e0b738f6052be99f90592feab23/tree-search/cookbook-md-tree/cookbook-tree-search/results.json",
  "retrieval_hit_ids": [
    "node-text-0fdecb35a49c0bf1af26897e9b8ef6c23ab0e501345338ae3ca2cc3e8409226a",
    "page-text-000000",
    "node-summary-0fdecb35a49c0bf1af26897e9b8ef6c23ab0e501345338ae3ca2cc3e8409226a"
  ],
  "search_mode": "deterministic",
  "selected_node_titles": [
    "Operating Handbook"
  ],
  "trace": [
    {
      "frontier_node_ids": [
        "0fdecb35a49c0bf1af26897e9b8ef6c23ab0e501345338ae3ca2cc3e8409226a"
      ],
      "selected_node_ids": [
        "0fdecb35a49c0bf1af26897e9b8ef6c23ab0e501345338ae3ca2cc3e8409226a"
      ],
      "selection_reason": "Selected frontier nodes by tree-search score: 0fdecb35a49c0bf1af26897e9b8ef6c23ab0e501345338ae3ca2cc3e8409226a=2.50",
      "step_index": 0,
      "termination_signal": "evidence_sufficient"
    }
  ]
}


## Section 11 — Preference-Aware Tree Search (Spec 07)


In [29]:
preference_tree_search_service = PreferenceAwareTreeSearchService(
    planner,
    retrieval_service,
    logger=runtime_logger,
    storage=pg_config,
)

base_policy_tree_search = tree_search_service.search(
    TreeSearchRequest(
        query="policies",
        tree_manifest_path=markdown_tree_manifest_path,
        retrieval_manifest_path=manifest_ref(
            run_type="retrieval",
            run_id=RUN_IDS["markdown_retrieval"],
            document_id=markdown_retrieval.document_id,
        ),
        search_run_id=f"{RUN_IDS['preference_tree_search']}-base",
        max_selected_nodes=1,
        retrieval_limit=5,
    )
)
preference_tree_search_response = preference_tree_search_service.search(
    PreferenceAwareTreeSearchRequest(
        query="policies",
        tree_manifest_path=markdown_tree_manifest_path,
        retrieval_manifest_path=manifest_ref(
            run_type="retrieval",
            run_id=RUN_IDS["markdown_retrieval"],
            document_id=markdown_retrieval.document_id,
        ),
        search_run_id=RUN_IDS["preference_tree_search"],
        max_selected_nodes=1,
        retrieval_limit=5,
        preference_snippets=(
            PreferenceSnippet(
                preference_id="prefer-litigation",
                scope=PreferenceScope.USER,
                text="Prefer litigation policies and case deadlines when policy sections tie.",
                priority=5,
            ),
        ),
    )
)

show_json(
    "Preference-aware tree search",
    {
        "base_selected_titles": [
            markdown_titles_by_id.get(candidate.node_id, candidate.node_id)
            for candidate in base_policy_tree_search.selected_nodes
        ],
        "preference_selected_titles": [
            markdown_titles_by_id.get(candidate.node_id, candidate.node_id)
            for candidate in preference_tree_search_response.selected_nodes
        ],
        "preference_selection": preference_tree_search_response.preference_selection.model_dump(mode="json"),
        "trace": [step.model_dump(mode="json") for step in preference_tree_search_response.trace],
        "results_ref": preference_tree_search_response.results_path,
    },
)


[TreeSearchStarted] query="policies" document=e386599943c9455ea1113d460e077bef156f7e0b738f6052be99f90592feab23 tree=cookbook-md-tree search=cookbook-preference-tree-search-base mode=deterministic max_depth=4
[RetrievalSearchStarted] query="policies" document=e386599943c9455ea1113d460e077bef156f7e0b738f6052be99f90592feab23 limit=5
[RetrievalSearchCompleted] query="policies" document=e386599943c9455ea1113d460e077bef156f7e0b738f6052be99f90592feab23 limit=5 candidates=3 hits=3 widened=false
[TreeSearchStepSelected] document=e386599943c9455ea1113d460e077bef156f7e0b738f6052be99f90592feab23 tree=cookbook-md-tree search=cookbook-preference-tree-search-base step=0 frontier=1 selected_nodes=1
[TreeSearchCompleted] document=e386599943c9455ea1113d460e077bef156f7e0b738f6052be99f90592feab23 tree=cookbook-md-tree search=cookbook-preference-tree-search-base mode=deterministic hits=3 selected=1
[PreferenceSelectionStarted] query="policies" search=cookbook-preference-tree-search limit=4 candidates=1
[Pr


Preference-aware tree search
{
  "base_selected_titles": [
    "Operating Handbook"
  ],
  "preference_selected_titles": [
    "Operating Handbook"
  ],
  "preference_selection": {
    "selected_snippets": [
      {
        "metadata": {},
        "preference_id": "prefer-litigation",
        "priority": 5,
        "scope": "user",
        "text": "Prefer litigation policies and case deadlines when policy sections tie."
      }
    ],
    "selection_reason": "Selected preference snippets by overlap and priority: prefer-litigation"
  },
  "results_ref": "pg://tree_search/cookbook-preference-tree-search/e386599943c9455ea1113d460e077bef156f7e0b738f6052be99f90592feab23/tree-search/cookbook-md-tree/cookbook-preference-tree-search/preference-aware-results.json",
  "trace": [
    {
      "applied_preference_ids": [
        "prefer-litigation"
      ],
      "frontier_node_ids": [
        "0fdecb35a49c0bf1af26897e9b8ef6c23ab0e501345338ae3ca2cc3e8409226a"
      ],
      "selected_node_ids": [


## Section 12 — Retrieval QA (self-contained NullVector path)


In [30]:
# RetrievalService was constructed with storage=pg_config (cell 20) to enable
# document-id-based search from Postgres. Here we pass corpus= directly (already
# loaded in-memory from cell 14), so the storage kwarg is not exercised in this path.
qa_query = "What do the litigation policies say about deadlines?"
retrieval_hits = retrieval_service.search(
    corpus=markdown_corpus,
    query=qa_query,
    limit=5,
)
qa_service = RetrievalQAService(retrieval_service, logger=runtime_logger)
qa_response = qa_service.answer(
    corpus=markdown_corpus,
    query=qa_query,
    limit=5,
)

show_json(
    "Retrieval QA",
    {
        "query": qa_query,
        "retrieval_hit_ids": [hit.unit.unit_id for hit in retrieval_hits],
        "answer_mode": qa_response.answer_mode,
        "answer": qa_response.answer,
        "citations": [citation.model_dump(mode="json") for citation in qa_response.citations],
    },
)

[RetrievalSearchStarted] query="What do the litigation policies say about deadlines?" document=e386599943c9455ea1113d460e077bef156f7e0b738f6052be99f90592feab23 limit=5
[RetrievalSearchCompleted] query="What do the litigation policies say about deadlines?" document=e386599943c9455ea1113d460e077bef156f7e0b738f6052be99f90592feab23 limit=5 candidates=13 hits=5 widened=false
[RetrievalQAStarted] query="What do the litigation policies say about deadlines?" document=e386599943c9455ea1113d460e077bef156f7e0b738f6052be99f90592feab23 limit=5
[RetrievalSearchStarted] query="What do the litigation policies say about deadlines?" document=e386599943c9455ea1113d460e077bef156f7e0b738f6052be99f90592feab23 limit=5
[RetrievalSearchCompleted] query="What do the litigation policies say about deadlines?" document=e386599943c9455ea1113d460e077bef156f7e0b738f6052be99f90592feab23 limit=5 candidates=13 hits=5 widened=false
[RetrievalQACompleted] query="What do the litigation policies say about deadlines?" docume


Retrieval QA
{
  "answer": "Litigation policies describe case deadlines, filing checkpoints, and escalation paths.",
  "answer_mode": "authoritative_text",
  "citations": [
    {
      "asset_path": null,
      "document_id": "e386599943c9455ea1113d460e077bef156f7e0b738f6052be99f90592feab23",
      "node_id": "f0773b053dbd186567bd8b0ec53600028fb583fdbb397a53a84b8ef3382cfe87",
      "page_label": "1",
      "page_span": {
        "end_page": 0,
        "start_page": 0
      },
      "quote": "Litigation policies describe case deadlines, filing checkpoints, and escalation paths.",
      "unit_id": "node-text-f0773b053dbd186567bd8b0ec53600028fb583fdbb397a53a84b8ef3382cfe87"
    },
    {
      "asset_path": null,
      "document_id": "e386599943c9455ea1113d460e077bef156f7e0b738f6052be99f90592feab23",
      "node_id": "4a07a70390a62d5da7c29ec9763cbcb56a8b20d0bf9c6aa530126d2feecf1546",
      "page_label": "1",
      "page_span": {
        "end_page": 0,
        "start_page": 0
      },
    

## Section 13 — Quickstart CLI (Spec 09)

The quickstart CLI stays intentionally thin: acquire, build a tree, and optionally build retrieval artifacts. The equivalent Postgres-backed command for the Markdown fixture created above is:

```bash
export NULLVECTOR_POSTGRES_CONNINFO='postgresql://REDACTED_DB_CRED@localhost:5432/nullvector'
python scripts/nullvector_quickstart.py           --source-path cookbook/_tmp/postgres_unified/spec05_operating_handbook.md           --storage-backend postgres           --pg-conninfo "$NULLVECTOR_POSTGRES_CONNINFO"           --build-retrieval           --print-tree-summary
```

Use the same command shape with the committed PDF fixture when you want the edge-only CLI path instead of the library walkthrough shown in this notebook.


In [31]:
# Uncomment to run the quickstart CLI directly from this notebook:
# !python scripts/nullvector_quickstart.py \
#     --source-path cookbook/_tmp/postgres_unified/spec05_operating_handbook.md \
#     --storage-backend postgres \
#     --pg-conninfo "$NULLVECTOR_POSTGRES_CONNINFO" \
#     --build-retrieval \
#     --print-tree-summary

## Section 14 — Results Inspection


In [32]:
summary = {
    "documents": {
        "pdf_document_id": pdf_acquisition.document_id,
        "markdown_document_id": markdown_acquisition.document_id,
    },
    "manifest_refs": {
        "pdf_acquisition": pdf_acquisition_manifest_path,
        "pdf_tree": pdf_tree_manifest_path,
        "pdf_retrieval": manifest_ref(
            run_type="retrieval",
            run_id=RUN_IDS["pdf_retrieval"],
            document_id=pdf_retrieval.document_id,
        ),
        "pdf_description": pdf_description_manifest_ref,
        "markdown_acquisition": markdown_acquisition_manifest_path,
        "markdown_tree": markdown_tree_manifest_path,
        "markdown_compaction": manifest_ref(
            run_type="tree_compaction",
            run_id=RUN_IDS["markdown_compaction"],
            document_id=markdown_compaction.document_id,
        ),
        "markdown_retrieval": manifest_ref(
            run_type="retrieval",
            run_id=RUN_IDS["markdown_retrieval"],
            document_id=markdown_retrieval.document_id,
        ),
        "markdown_description": markdown_description_manifest_ref,
    },
    "selection_winners": {
        "metadata": [candidate.document_id for candidate in metadata_response.candidates],
        "description": [candidate.document_id for candidate in description_selection_response.candidates],
        "semantic_prefilter": [hit.document_id for hit in semantic_prefilter_response.hits],
    },
    "pdf_coverage": pdf_tree_coverage,
    "tree_search": {
        "selected_titles": [
            markdown_titles_by_id.get(candidate.node_id, candidate.node_id)
            for candidate in tree_search_response.selected_nodes
        ],
        "preference_selected_titles": [
            markdown_titles_by_id.get(candidate.node_id, candidate.node_id)
            for candidate in preference_tree_search_response.selected_nodes
        ],
    },
    "qa_answer": qa_response.answer,
    "observability": {
        "enabled": runtime_logger is not None,
        "jsonl_path": OBSERVABILITY_JSONL_PATH,
    },
}
show_json("Unified Postgres cookbook summary", summary)



Unified Postgres cookbook summary
{
  "documents": {
    "markdown_document_id": "e386599943c9455ea1113d460e077bef156f7e0b738f6052be99f90592feab23",
    "pdf_document_id": "798d2f27d45d2ccda3694005c2ed60bc0b413b8b299f3a5d4ade7c5867094896"
  },
  "manifest_refs": {
    "markdown_acquisition": "pg://acquisition/cookbook-md-acquisition/e386599943c9455ea1113d460e077bef156f7e0b738f6052be99f90592feab23/manifest.json",
    "markdown_compaction": "pg://tree_compaction/cookbook-md-compaction/e386599943c9455ea1113d460e077bef156f7e0b738f6052be99f90592feab23/manifest.json",
    "markdown_description": "pg://document_description/cookbook-md-description/e386599943c9455ea1113d460e077bef156f7e0b738f6052be99f90592feab23/manifest.json",
    "markdown_retrieval": "pg://retrieval/cookbook-md-retrieval/e386599943c9455ea1113d460e077bef156f7e0b738f6052be99f90592feab23/manifest.json",
    "markdown_tree": "pg://tree/cookbook-md-tree/e386599943c9455ea1113d460e077bef156f7e0b738f6052be99f90592feab23/manifest.js

## Notes

- This notebook is designed for structural correctness and manual execution against a live PostgreSQL database; it is not executed in CI.
- The Markdown source is generated locally inside `cookbook/_tmp/postgres_unified/` so Spec 05 can be demonstrated alongside the committed PDF fixture. This directory is gitignored — no accidental commits of generated fixtures.
- The live summarized-tree path uses notebook-local Groq rotation helpers. Notebook logging only prints masked key labels such as `slot-02 (gsk_...abcd)`.
- NullVector runtime observability is now on by default for this walkthrough, so ingest, tree build, selection, retrieval, and QA cells emit progress lines and append structured events to `artifacts/observability/nullvector-events.jsonl` unless you override the destination with `NULLVECTOR_OBSERVABILITY_JSONL_PATH`.
- The PDF coverage summary now distinguishes ingestion from tree attachment. `903000608.pdf` still ingests all 148 pages, but page 0 remains a visual-only `before_first_heading` gap and is preserved in retrieval through `unassigned_span` and `visual` units.
- The collection-selection block is split into three subsections (9a, 9b, 9c) corresponding to Specs 02, 03, and 04 so each strategy's inputs and outputs are independently visible.
- The quickstart CLI remains intentionally narrower than the library walkthrough: it covers acquisition, tree build, and optional retrieval only. A commented-out executable cell is provided for copy-paste convenience.